# Notebook 2: Fine-tuning Pretraining vs. No Pretraining

### Pipeline A (with pre-training): Load final pre-trained model -> fine-tune on bug fixing
### Pipeline B (without pre-training): Initialize fresh T5-small from scratch -> fine-tune on bug fixing

In [ ]:
# !pip install datasets transformers sentencepiece tokenizers accelerate torch faiss-cpu sentence-transformers --quiet

In [3]:
!python --version

Python 3.10.12


In [4]:
import torch
import os
from datasets import load_dataset
from transformers import (
    PreTrainedTokenizerFast,
    T5Config,
    T5ForConditionalGeneration,
    Trainer,
    TrainingArguments,
    DataCollatorForSeq2Seq,
)

In [5]:
# Create Directories

dirs = [
    "./pipeline_a_finetuned",   # Pipeline A output
    "./pipeline_b_finetuned",   # Pipeline B output
]

for d in dirs:
    os.makedirs(d, exist_ok=True)
    print(f"Created: {d}")

print("\nAll directories ready.")

Created: ./pipeline_a_finetuned
Created: ./pipeline_b_finetuned

All directories ready.


In [6]:
# Load Tokenizer (Same tokenizer for both pipelines)
# Same tokenizer trained on the pre-training corpus to keep consistency

tokenizer = PreTrainedTokenizerFast.from_pretrained("./java_tokenizer")
print(f"Tokenizer loaded from ./java_tokenizer  |  Vocab size: {len(tokenizer)}")

# Confirm sentinel tokens are loaded
sentinel_id = tokenizer.convert_tokens_to_ids("<extra_id_0>")
assert sentinel_id != tokenizer.unk_token_id, \
    "ERROR: Sentinel tokens missing from loaded tokenizer!"
print(f"<extra_id_0> ID: {sentinel_id}... loaded")

Tokenizer loaded from ./java_tokenizer  |  Vocab size: 16384
<extra_id_0> ID: 3... loaded


In [7]:
# Load Fine-tuning Dataset CodeXGlue Code Refinement medium split

# load_dataset("google/code_x_glue_cc_code_refinement", name="medium")
# Provides ~52K train, ~6.5K validation, ~6.5K test pairs.
# Each sample has "buggy" and "fixed" fields.

print("\nLoading CodeXGlue Code Refinement (medium)...")
dataset = load_dataset("google/code_x_glue_cc_code_refinement", name="medium")
print(dataset)
print(f"\nTrain samples    : {len(dataset['train'])}")
print(f"Validation samples: {len(dataset['validation'])}")
print(f"Test samples      : {len(dataset['test'])}")

# Loading check print one example
print("\n--- Example from training set ---")
print("Buggy :", dataset["train"][0]["buggy"][:120])
print("Fixed :", dataset["train"][0]["fixed"][:120])



Loading CodeXGlue Code Refinement (medium)...


README.md: 0.00B [00:00, ?B/s]

medium/train-00000-of-00001.parquet:   0%|          | 0.00/11.9M [00:00<?, ?B/s]

medium/validation-00000-of-00001.parquet:   0%|          | 0.00/1.50M [00:00<?, ?B/s]

medium/test-00000-of-00001.parquet:   0%|          | 0.00/1.49M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/52364 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/6546 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6545 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'buggy', 'fixed'],
        num_rows: 52364
    })
    validation: Dataset({
        features: ['id', 'buggy', 'fixed'],
        num_rows: 6546
    })
    test: Dataset({
        features: ['id', 'buggy', 'fixed'],
        num_rows: 6545
    })
})

Train samples    : 52364
Validation samples: 6546
Test samples      : 6545

--- Example from training set ---
Buggy : public static TYPE_1 init ( java.lang.String name , java.util.Date date ) { TYPE_1 VAR_1 = new TYPE_1 ( ) ; VAR_1 . METH
Fixed : public static TYPE_1 init ( java.lang.String name , java.util.Date date ) { TYPE_1 VAR_1 = new TYPE_1 ( ) ; VAR_1 . METH


In [ ]:
# Preprocessing
# Tokenize buggy Input and fixed Target

# Input format: "fix: <buggy method>" , task prefix following T5 convention
# Labels: tokenized fixed method with padding replaced by -100 so the loss
#         function ignores padding positions

def preprocess_fn(examples):
    inputs = ["fix: " + b for b in examples["buggy"]]
    targets = examples["fixed"]

    # Tokenize inputs and targets in a single call
    model_inputs = tokenizer(
        inputs,
        text_target=targets, 
        max_length=512,
        truncation=True,
        padding=False,      # Padding handled by DataCollatorForSeq2Seq
    )

    return model_inputs


print("\nTokenizing dataset...")
tokenized_dataset = dataset.map(
    preprocess_fn,
    batched=True,
    remove_columns=dataset["train"].column_names,
)
print("Tokenization complete.")
print(f"Train : {len(tokenized_dataset['train'])} samples")
print(f"Val   : {len(tokenized_dataset['validation'])} samples")
print(f"Test  : {len(tokenized_dataset['test'])} samples")



Tokenizing dataset...


Map:   0%|          | 0/52364 [00:00<?, ? examples/s]

Map:   0%|          | 0/6546 [00:00<?, ? examples/s]

Map:   0%|          | 0/6545 [00:00<?, ? examples/s]

Tokenization complete.
Train : 52364 samples
Val   : 6546 samples
Test  : 6545 samples


In [10]:
# Verify the preprocessing ran correctly
# Checks: input IDs decode back to readable text, labels are set, no -100 leakage.

sample = tokenized_dataset["train"][0]

print("--- Tokenized Example (train[0]) ---")
print(f"\nInput IDs  ({len(sample['input_ids'])} tokens):")
print(sample["input_ids"])

print(f"\nDecoded input:")
print(tokenizer.decode(sample["input_ids"], skip_special_tokens=True))

print(f"\nLabel IDs  ({len(sample['labels'])} tokens):")
print(sample["labels"])

# Labels should not contain -100 yet — padding handled by collator at batch time
print(f"\nDecoded labels:")
print(tokenizer.decode(sample["labels"], skip_special_tokens=True))

print(f"\nAttention mask: {sample['attention_mask'][:20]} ...")
print("\nTokenize Good!" if len(sample["input_ids"]) > 0 and len(sample["labels"]) > 0
      else "WARNING: Empty input or labels detected!")


--- Tokenized Example (train[0]) ---

Input IDs  (111 tokens):
[3621, 136, 124, 157, 105, 469, 1923, 914, 112, 545, 103, 1142, 103, 132, 250, 105, 107, 545, 103, 638, 103, 475, 992, 196, 108, 105, 469, 1923, 105, 2173, 1923, 110, 118, 105, 469, 1923, 112, 196, 105, 114, 105, 2173, 1923, 105, 103, 6730, 1923, 112, 250, 196, 105, 114, 545, 103, 638, 103, 932, 105, 2173, 113, 180, 110, 545, 103, 638, 103, 932, 103, 861, 112, 196, 105, 114, 105, 2173, 113, 180, 105, 103, 6730, 113, 180, 112, 992, 196, 105, 114, 105, 2173, 1923, 105, 103, 6730, 113, 258, 112, 105, 2173, 113, 180, 196, 105, 114, 117, 105, 2173, 1923, 105, 114, 106, 2]

Decoded input:
fix: public static TYPE_1 init ( java.lang.String name , java.util.Date date ) { TYPE_1 VAR_1 = new TYPE_1 ( ) ; VAR_1 . METHOD_1 ( name ) ; java.util.Calendar VAR_2 = java.util.Calendar.getInstance ( ) ; VAR_2 . METHOD_2 ( date ) ; VAR_1 . METHOD_3 ( VAR_2 ) ; return VAR_1 ; }

Label IDs  (125 tokens):
[124, 157, 105, 469, 1923, 914, 112, 545, 

In [14]:
# Fine-tuning Function

# Use the same fine-tuning hyperparameters for both pipelines
# so the only variable is whether the model was pre-trained.
# Both pipelines will call this function

def run_finetuning(model_instance, output_dir):
    """
    Fine-tune model_instance on bug fixing and save to output_dir.
    Same hyperparameters used for Pipeline A and Pipeline B.
    """
    # DataCollatorForSeq2Seq handles dynamic padding and sets pad positions
    # in labels to -100 so loss is not computed on padding tokens
    collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model_instance,
        label_pad_token_id=-100,    # Ignore padding in loss computation
        pad_to_multiple_of=8,       # Efficient on tensor cores
    )

    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=10,            # Set it higher than you think you need
        greater_is_better=False,        # Lower loss is better
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        learning_rate=5e-5,
        warmup_steps=200,
        weight_decay=0.01,
        eval_strategy="epoch",          # Evaluate after each epoch
        save_strategy="epoch",          # Save after each epoch
        save_total_limit=1,             # Keep only best checkpoint on disk
        load_best_model_at_end=True,    # Load best val-loss model at end
        metric_for_best_model="eval_loss",
        logging_steps=100,
        logging_strategy="steps",
        fp16=torch.cuda.is_available(),
        dataloader_num_workers=2,
        report_to="none",
    )

    trainer = Trainer(
        model=model_instance,
        args=training_args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["validation"],
        data_collator=collator,
    )

    trainer.train()

    # Save final model and tokenizer together for evaluation notebook
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f"\nModel and tokenizer saved to {output_dir}")

    return trainer


In [ ]:
# Pipeline A Load Pre-trained Model -> Fine-tune
# "Load the final pre-trained model and fine-tune on bug fixing."
# Weights, embeddings, and config all come from the pre-training checkpoint.

print("\n" + "-"*60)
print("PIPELINE A — With Pre-training")
print("-"*60)

model_a = T5ForConditionalGeneration.from_pretrained("./final_pretrained_model")
print(f"Pre-trained model loaded from ./final_pretrained_model")
print(f"Embedding shape: {model_a.shared.weight.shape}")

trainer_a = run_finetuning(model_a, "./pipeline_a_finetuned")

# Log epoch-level validation losses for the report
print("\nPipeline A — Validation loss per epoch:")
for entry in trainer_a.state.log_history:
    if "eval_loss" in entry:
        print(f"  Epoch {entry['epoch']:.0f}  |  eval_loss: {entry['eval_loss']:.4f}")


------------------------------------------------------------
PIPELINE A — With Pre-training
------------------------------------------------------------


Loading weights:   0%|          | 0/143 [00:00<?, ?it/s]

Pre-trained model loaded from ./final_pretrained_model
Embedding shape: torch.Size([16384, 512])


Epoch,Training Loss,Validation Loss
1,0.572512,0.510937
2,0.473865,0.415575
3,0.424632,0.385350
4,0.388869,0.362220
5,0.347440,0.327884
6,0.317552,0.287469
7,0.295814,0.258962
8,0.262179,0.239515
9,0.246866,0.223784
10,0.232283,0.215677


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model and tokenizer saved to ./pipeline_a_finetuned

Pipeline A — Validation loss per epoch:
  Epoch 1  |  eval_loss: 0.5109
  Epoch 2  |  eval_loss: 0.4156
  Epoch 3  |  eval_loss: 0.3853
  Epoch 4  |  eval_loss: 0.3622
  Epoch 5  |  eval_loss: 0.3279
  Epoch 6  |  eval_loss: 0.2875
  Epoch 7  |  eval_loss: 0.2590
  Epoch 8  |  eval_loss: 0.2395
  Epoch 9  |  eval_loss: 0.2238
  Epoch 10  |  eval_loss: 0.2157


In [ ]:
# Pipeline B Initialize from Scratch -> Fine-tune (No Pre-training)

# "Initialize a fresh T5-small from scratch (same T5Config,
# same tokenizer, same resize_token_embeddings call) and fine-tune directly
# on bug fixing. This model has never seen the pre-training corpus."
#
# Manually defined T5Config


print("\n" + "-"*60)
print("PIPELINE B — Without Pre-training (Initialized from Scratch)")
print("-"*60)

t5_config = T5Config(
    decoder_start_token_id=0,       # <pad>
    pad_token_id=0,
    eos_token_id=1,
    vocab_size=len(tokenizer),      # Same vocab as pre-training
    d_model=512,                    # T5-small
    d_ff=2048,
    d_kv=64,
    num_heads=8,
    num_layers=6,
    num_decoder_layers=6,
    feed_forward_proj="gated-gelu"
)

model_b = T5ForConditionalGeneration(config=t5_config)
model_b.resize_token_embeddings(len(tokenizer))  # same as pre-training setup

print(f"Fresh model initialized from scratch.")
print(f"Embedding shape: {model_b.shared.weight.shape}")
total_params = sum(p.numel() for p in model_b.parameters())
print(f"Total parameters: {total_params:,}  ")

trainer_b = run_finetuning(model_b, "./pipeline_b_finetuned")

# Log epoch-level validation losses for the report
print("\nPipeline B — Validation loss per epoch:")
for entry in trainer_b.state.log_history:
    if "eval_loss" in entry:
        print(f"  Epoch {entry['epoch']:.0f}  |  eval_loss: {entry['eval_loss']:.4f}")



------------------------------------------------------------
PIPELINE B — Without Pre-training (Initialized from Scratch)
------------------------------------------------------------
Fresh model initialized from scratch.
Embedding shape: torch.Size([16384, 512])
Total parameters: 65,028,608  


Epoch,Training Loss,Validation Loss
1,0.564410,0.493630
2,0.463708,0.411221
3,0.413028,0.376983
4,0.371920,0.345892
5,0.324961,0.307561
6,0.294169,0.264864
7,0.271783,0.238146
8,0.240841,0.220554
9,0.221809,0.204355
10,0.208396,0.197253


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model and tokenizer saved to ./pipeline_b_finetuned

Pipeline B — Validation loss per epoch:
  Epoch 1  |  eval_loss: 0.4936
  Epoch 2  |  eval_loss: 0.4112
  Epoch 3  |  eval_loss: 0.3770
  Epoch 4  |  eval_loss: 0.3459
  Epoch 5  |  eval_loss: 0.3076
  Epoch 6  |  eval_loss: 0.2649
  Epoch 7  |  eval_loss: 0.2381
  Epoch 8  |  eval_loss: 0.2206
  Epoch 9  |  eval_loss: 0.2044
  Epoch 10  |  eval_loss: 0.1973


In [17]:
# Confirm Both Pipelines Completed

print("FINE-TUNING COMPLETE...")

print("Pipeline A checkpoint : ./pipeline_a_finetuned")
print("Pipeline B checkpoint : ./pipeline_b_finetuned")
print("\nBoth checkpoints include the tokenizer.")
print("Load them in Notebook 3 for CodeBLEU and exact match evaluation.")

# Quick parameter count confirmation
params_a = sum(p.numel() for p in model_a.parameters())
params_b = sum(p.numel() for p in model_b.parameters())
print(f"\nPipeline A parameters: {params_a:,}")
print(f"Pipeline B parameters: {params_b:,}")
assert params_a == params_b, "ERROR: Model sizes differ!"
print("Parameter count match confirmed")

FINE-TUNING COMPLETE...
Pipeline A checkpoint : ./pipeline_a_finetuned
Pipeline B checkpoint : ./pipeline_b_finetuned

Both checkpoints include the tokenizer.
Load them in Notebook 3 for CodeBLEU and exact match evaluation.

Pipeline A parameters: 65,028,608
Pipeline B parameters: 65,028,608
Parameter count match confirmed


Output File Directories:

Pipeline A Checkpoint files:
- `./pipeline_a_finetuned`   

Pipeline B Checkpoint files:
- `./pipeline_b_finetuned`